# Preprocessing: Data Loading, Tokenization & Vectorization
Run this notebook once before training.

In [1]:
import pandas as pd
import torch
import sentencepiece as spm
import os
from tqdm import tqdm

# Config
MAX_LEN = 96
VOCAB_SIZE = 22000
TRAIN_RATIO = 0.995
RANDOM_SEED = 42

In [2]:
# Load data and split
df = pd.read_parquet('data/HF_dataset_500k.parquet')
df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
train_size = int(TRAIN_RATIO * len(df))
df_train = df[:train_size]
df_test = df[train_size:]
print(f'Train: {len(df_train)}, Test: {len(df_test)}')

Train: 497500, Test: 2500


In [3]:
# Train SentencePiece tokenizer
corpus_path = 'processed/corpus.txt'
os.makedirs('processed', exist_ok=True)
os.makedirs('tokenizer', exist_ok=True)

with open(corpus_path, 'w', encoding='utf-8') as f:
    for _, row in df.iterrows():
        f.write(row['ru'] + '\n')
        f.write(row['tat'] + '\n')

spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix='tokenizer/spm',
    vocab_size=VOCAB_SIZE,
    model_type='unigram',
    character_coverage=1.0,
    unk_id=0,
    bos_id=1,
    eos_id=2,
    pad_id=3,
)

# Cleanup temp corpus file
os.remove(corpus_path)
print('Tokenizer trained and saved to tokenizer/')

Tokenizer trained and saved to tokenizer/


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: processed/corpus.txt
  input_format: 
  model_prefix: tokenizer/spm
  model_type: UNIGRAM
  vocab_size: 22000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: 3
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0

In [4]:
# Tokenize and save
sp = spm.SentencePieceProcessor(model_file='tokenizer/spm.model')

def encode_row(row, sp):
    vocab_size = sp.get_piece_size()
    unk_id = sp.unk_id()
    bos_id = sp.bos_id()
    eos_id = sp.eos_id()

    src_tokens = [tok if tok < vocab_size else unk_id for tok in sp.encode(row['ru'])]
    src = [bos_id] + src_tokens[:MAX_LEN-2] + [eos_id]

    tgt_tokens = [tok if tok < vocab_size else unk_id for tok in sp.encode(row['tat'])]
    tgt = [bos_id] + tgt_tokens[:MAX_LEN-2] + [eos_id]

    return {
        'src': torch.tensor(src, dtype=torch.long),
        'tgt': torch.tensor(tgt, dtype=torch.long)
    }

os.makedirs('preprocessed', exist_ok=True)

print('Tokenizing train data...')
tokenized_train = [encode_row(row, sp) for _, row in tqdm(df_train.iterrows(), total=len(df_train))]

print('Tokenizing test data...')
tokenized_test = [encode_row(row, sp) for _, row in tqdm(df_test.iterrows(), total=len(df_test))]

torch.save(tokenized_train, 'preprocessed/train_tokenized.pt')
torch.save(tokenized_test, 'preprocessed/test_tokenized.pt')
print(f'Saved to preprocessed/ ({len(tokenized_train)} train, {len(tokenized_test)} test)')

Tokenizing train data...


100%|█████████████████████████████████████████████████████████████████████████| 497500/497500 [01:18<00:00, 6317.07it/s]


Tokenizing test data...


100%|█████████████████████████████████████████████████████████████████████████████| 2500/2500 [00:00<00:00, 7038.67it/s]


Saved to preprocessed/ (497500 train, 2500 test)
